In [24]:
from WindPy import w
import numpy as np
import pandas as pd
import datetime
import os
import json
from tqdm import tqdm

w.start()
w.isconnected()

True

In [2]:
with open("INDEX_START.json", "r", encoding="utf-8") as f:
    INDEX_START = json.load(f)

# Load Local Database

In [20]:
INDEX_DATA = {}

for ticker in INDEX_START:
    path = f"Data/{ticker}.csv"
    
    if os.path.isfile(path): 
        data = pd.read_csv(path, index_col = 0, parse_dates = True)
        data.index = data.index.date
        INDEX_DATA[ticker] = data.copy(deep = True)
        
len(INDEX_DATA)

30

# Update Existing Tickers

In [23]:
today = "2025-02-18"

for ticker in tqdm(INDEX_DATA):
    data = INDEX_DATA[ticker]
    start = (data.index[-1] + pd.Timedelta(days = 1)).strftime("%Y-%m-%d")
    
    new_data = w.wsd(ticker, 'pe_ttm,dividendyield2', start, today, '', usedf = True)[1]
    
    if new_data.columns[0] == 'OUTMESSAGE':
        continue
    
    INDEX_DATA[ticker] = pd.concat([data, new_data])
    INDEX_DATA[ticker].dropna(inplace = True)

# Download New Tickers

In [31]:
for ticker, start in tqdm(INDEX_START.items()):
    if ticker in INDEX_DATA:
        continue
        
    assert ticker not in INDEX_DATA
    
    data = w.wsd(ticker, 'pe_ttm,dividendyield2', start, today, '', usedf = True)[1]
    
    junk = data['PE_TTM']
    
    data.dropna(inplace = True)
    
    INDEX_DATA[ticker] = data.copy(deep = True)

KeyError: 'PE_TTM'

In [33]:
len(INDEX_DATA)

45

In [18]:
for ticker, start in INDEX_START.items():
    data = w.wsd(ticker, 'pe_ttm,dividendyield2', start, today, '', usedf = True)[1]

    INDEX_DATA[ticker] = data.copy(deep = True)

datetime.date

In [ ]:
ticker = next(gen)
print(ticker)
INDEX_DATA[ticker]

In [ ]:
for ticker in INDEX_START:
    INDEX_DATA[ticker].dropna(inplace = True)

In [34]:
for ticker, data in INDEX_DATA.items():
    path = os.path.join("Data", f"{ticker}.csv")
    data.to_csv(path, index = True, encoding = "utf-8")

In [ ]:
ticker = 